<hr style="color:#808080;">
<p align="center"><b>N E P T U N E</b></p>
<hr style="color:#808080;">

In [1]:
# ---------- Libraries ----------
#
import math
import neptune
import torch

from pathlib import Path
from shaggy.loss import loss_geometry_embedding

from neptune.data import DATASET_CONDITIONING, DATASET_VARIABLES_OCEAN, DATASET_VARIABLES_SURFACE, X, Y, Z
from neptune.model import OceanEncoder, SurfaceEncoder
from neptune.tools import load_configuration

# ---------- Jupyter ----------
#
%reload_ext autoreload
%autoreload 2

# ---------- Device ----------
#
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

/gpfs/home/acad/ulg-mast/vmangele/micromamba/envs/neptune/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


<hr style="color:#808080;">
<p align="center"><b>C O N F I G U R A T I O N</b></p>
<hr style="color:#808080;">

In [7]:
# ---------- Configuration | Encoders of experiments/forecast-latent-states/configs/encoders.yml (rerun after each change) ----------
#
PATH_CONFIG = Path(neptune.__file__).parents[1] / "experiments" / "forecast-latent-states" / "configs" / "encoders.yml"
config      = load_configuration(PATH_CONFIG)[0]

# Encoders | States (C, L_1, ..., L_S) and conditioning channels, the date (sin, cos) and ERA5 at the surface, the date in the ocean
ENCODERS     = {"surface": SurfaceEncoder, "ocean": OceanEncoder}
STATES       = {"surface": (len(DATASET_VARIABLES_SURFACE), Y, X), "ocean": (len(DATASET_VARIABLES_OCEAN), Z, Y, X)}
CONDITIONING = {"surface": 2 + len(DATASET_CONDITIONING), "ocean": 2}


def build(group: str) -> torch.nn.Module:
    r"""Build the encoder of a group of variables from the configuration."""
    return ENCODERS[group](lat_channels=config["Model"]["lat_channels"], **config["Model"][f"config_{group}"])


print(f"Configuration : {PATH_CONFIG}")

Configuration : /gpfs/home/acad/ulg-mast/vmangele/neptune/experiments/forecast-latent-states/configs/encoders.yml


In [8]:
# ---------- Compression | Latent representation of each encoder, against the size of the states ----------
#
for group in ENCODERS:
    encoder    = build(group)
    states     = math.prod(STATES[group])
    latent     = encoder.lat_channels * encoder.tokens
    parameters = sum(p.numel() for p in encoder.parameters())

    print(f"{group.capitalize()}")
    print(f"  States      : {STATES[group]} = {states:,} values")
    print(f"  Latent      : {(encoder.lat_channels, *encoder.shape)} = {latent:,} values ({encoder.tokens} tokens of {encoder.lat_channels} channels)")
    print(f"  Compression : ×{states / latent:.1f}")
    print(f"  Parameters  : {parameters / 1e6:.1f} M")

Surface
  States      : (3, 256, 576) = 442,368 values
  Latent      : (256, 16, 36) = 147,456 values (576 tokens of 256 channels)
  Compression : ×3.0
  Parameters  : 6.9 M
Ocean
  States      : (10, 48, 256, 576) = 70,778,880 values
  Latent      : (256, 6, 32, 72) = 3,538,944 values (13824 tokens of 256 channels)
  Compression : ×20.0
  Parameters  : 25.2 M


<hr style="color:#808080;">
<p align="center"><b>M E M O R Y</b></p>
<hr style="color:#808080;">

In [4]:
# ---------- Memory | Largest batch per GPU of a training step (bf16 forward, GME loss and gradient), doubled until out of memory ----------
#
BATCH_MAX = 1024  # Stops the search, the surface encoder being light


def peak_memory(group: str, batch: int) -> float:
    r"""Peak GPU memory of a training step of the encoder of a group of variables, for a batch [GB]."""

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    encoder = build(group).to(DEVICE)
    states  = torch.randn(batch, *STATES[group], device=DEVICE)
    cond    = torch.randn(batch, CONDITIONING[group], *STATES[group][1:], device=DEVICE)

    with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
        z = encoder(torch.cat([states, cond], dim=1))

    loss_geometry_embedding(states, z.float()).backward()

    return torch.cuda.max_memory_allocated() / 1e9


total = torch.cuda.get_device_properties(DEVICE).total_memory / 1e9
print(f"GPU : {torch.cuda.get_device_name(DEVICE)} ({total:.0f} GB)\n")

for group in ENCODERS:
    batch, largest = 2, None
    while batch <= BATCH_MAX:
        try:
            memory = peak_memory(group, batch)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            break
        print(f"{group.capitalize():<8} | batch {batch:<4} | peak {memory:6.2f} GB ({100 * memory / total:3.0f} %)")
        batch, largest = 2 * batch, batch

    print(f"{group.capitalize():<8} | largest batch per GPU : {largest}\n")

print("Not included : the optimizer states (SOAP) and the DDP buckets, of the order of a few times the parameters, below 1 GB here")

GPU : NVIDIA A100-SXM4-40GB (42 GB)

Surface  | batch 2    | peak   0.25 GB (  1 %)
Surface  | batch 4    | peak   0.43 GB (  1 %)
Surface  | batch 8    | peak   0.80 GB (  2 %)
Surface  | batch 16   | peak   1.85 GB (  4 %)
Surface  | batch 32   | peak   2.92 GB (  7 %)
Surface  | batch 64   | peak   5.75 GB ( 14 %)
Surface  | batch 128  | peak  11.41 GB ( 27 %)
Surface  | batch 256  | peak  26.97 GB ( 64 %)
Surface  | largest batch per GPU : 256

Ocean    | batch 2    | peak   8.99 GB ( 21 %)
Ocean    | batch 4    | peak  17.52 GB ( 41 %)
Ocean    | batch 8    | peak  34.58 GB ( 81 %)
Ocean    | largest batch per GPU : 8

Not included : the optimizer states (SOAP) and the DDP buckets, of the order of a few times the parameters, below 1 GB here
